In [46]:
import networkx as nx
import folium
from math import radians, sin, cos, sqrt, atan2

diem_dau = [10.76135555, 106.66834593]
diem_cuoi = [10.78021171, 106.69887296]

nut = {
    'UEH': [10.76135555, 106.66834593],
    'Nguyen Tri Phuong': [10.75906700, 106.66902900],
    'An Duong Vuong': [10.75787700, 106.67512000],
    'Cong Quynh': [10.76704300, 106.68747800],
    'Ben Thanh': [10.77252069, 106.69801918],
    'Nha tho Duc Ba': [10.78021171, 106.69887296],

    'Cach Mang Thang Tam': [10.77465600, 106.68756100],
    'Dien Bien Phu': [10.78750800, 106.69474800],
    'Vo Thi Sau': [10.78508500, 106.68962700],
    'Nam Ky Khoi Nghia': [10.78677400, 106.68729500]
}

def khoang_cach_thang(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

canh_tam = [
    ['UEH', 'Nguyen Tri Phuong'],
    ['Nguyen Tri Phuong', 'An Duong Vuong'],
    ['An Duong Vuong', 'Cong Quynh'],
    ['Cong Quynh', 'Ben Thanh'],
    ['Ben Thanh', 'Nha tho Duc Ba'],

    ['UEH', 'Cach Mang Thang Tam'],
    ['Cach Mang Thang Tam', 'Dien Bien Phu'],
    ['Dien Bien Phu', 'Vo Thi Sau'],
    ['Vo Thi Sau', 'Nam Ky Khoi Nghia'],
    ['Nam Ky Khoi Nghia', 'Nha tho Duc Ba']
]

canh = []

for mot_canh in canh_tam:
    nut_1 = mot_canh[0]
    nut_2 = mot_canh[1]

    do_dai = round(
        khoang_cach_thang(
            nut[nut_1][0],
            nut[nut_1][1],
            nut[nut_2][0],
            nut[nut_2][1]
        )
    )

    if do_dai > 10000:
        raise ValueError('Khoang cach vuot qua 10km: ' + nut_1 + ' - ' + nut_2)

    canh.append([nut_1, nut_2, do_dai])

G = nx.Graph()

for ten_nut in nut:
    G.add_node(ten_nut, vi_do=nut[ten_nut][0], kinh_do=nut[ten_nut][1])

for mot_canh in canh:
    nut_1 = mot_canh[0]
    nut_2 = mot_canh[1]
    do_dai = mot_canh[2]

    G.add_edge(nut_1, nut_2, weight=do_dai)

def heuristic(u, v):
    lat1 = G.nodes[u]['vi_do']
    lon1 = G.nodes[u]['kinh_do']
    lat2 = G.nodes[v]['vi_do']
    lon2 = G.nodes[v]['kinh_do']

    return khoang_cach_thang(lat1, lon1, lat2, lon2)

duong_dijkstra = nx.shortest_path(G, 'UEH', 'Nha tho Duc Ba', weight='weight', method='dijkstra')
do_dai_dijkstra = nx.shortest_path_length(G, 'UEH', 'Nha tho Duc Ba', weight='weight')

duong_astar = nx.astar_path(G, 'UEH', 'Nha tho Duc Ba', heuristic=heuristic, weight='weight')
do_dai_astar = nx.path_weight(G, duong_astar, weight='weight')

print('Duong di bang Dijkstra:')
print(duong_dijkstra)
print('Do dai Dijkstra:')
print(do_dai_dijkstra, 'm')

print('Duong di bang A*:')
print(duong_astar)
print('Do dai A*:')
print(do_dai_astar, 'm')

if duong_dijkstra == duong_astar:
    print('Hai thuat toan cho ra cung mot duong di')
else:
    print('Hai thuat toan cho ra hai duong di khac nhau')

def lay_toa_do(duong):
    toa_do = []

    for ten_nut in duong:
        vi_do = G.nodes[ten_nut]['vi_do']
        kinh_do = G.nodes[ten_nut]['kinh_do']
        toa_do.append([vi_do, kinh_do])

    return toa_do

m = folium.Map(location=diem_dau, zoom_start=14)

for mot_canh in canh:
    nut_1 = mot_canh[0]
    nut_2 = mot_canh[1]

    toa_do_canh = [
        [nut[nut_1][0], nut[nut_1][1]],
        [nut[nut_2][0], nut[nut_2][1]]
    ]

    folium.PolyLine(toa_do_canh, color='gray', weight=2, opacity=0.4).add_to(m)

folium.Marker(diem_dau, popup='Diem xuat phat - UEH', tooltip='click').add_to(m)

folium.Marker(diem_cuoi, popup='Diem den - Nha tho Duc Ba', tooltip='click').add_to(m)

for ten_nut in nut:
    folium.CircleMarker(
        location=[nut[ten_nut][0], nut[ten_nut][1]],
        radius=4,
        color='black',
        fill=True,
        fill_color='black',
        popup=ten_nut
    ).add_to(m)

folium.PolyLine(
    lay_toa_do(duong_dijkstra),
    color='blue',
    weight=6,
    opacity=0.8,
    popup='Dijkstra - duong ngan nhat, dai ' + str(do_dai_dijkstra) + ' m'
).add_to(m)

folium.PolyLine(
    lay_toa_do(duong_astar),
    color='red',
    weight=4,
    opacity=0.9,
    dash_array='10',
    popup='A* - dai ' + str(do_dai_astar) + ' m'
).add_to(m)

m


Duong di bang Dijkstra:
['UEH', 'Nguyen Tri Phuong', 'An Duong Vuong', 'Cong Quynh', 'Ben Thanh', 'Nha tho Duc Ba']
Do dai Dijkstra:
4798 m
Duong di bang A*:
['UEH', 'Nguyen Tri Phuong', 'An Duong Vuong', 'Cong Quynh', 'Ben Thanh', 'Nha tho Duc Ba']
Do dai A*:
4798 m
Hai thuat toan cho ra cung mot duong di
